In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Loading  cleaned data
df = pd.read_csv('yoga_poses_cleaned_final.csv')

#  Encoding labels (Turn 'Plank' into 0, 'Tree' into 1, etc.)
encoder = LabelEncoder()
y = encoder.fit_transform(df['pose_label'])
X = df.drop('pose_label', axis=1).values

# Scaling 
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Reshaping X for 1D CNN: (samples, features, 1)
X = X.reshape(X.shape[0], X.shape[1], 1)

# Splitting into Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Building the 1D CNN Model
model = models.Sequential([
    layers.Conv1D(64, kernel_size=3, activation='relu', input_shape=(X.shape[1], 1)),
    layers.MaxPooling1D(pool_size=2),
    layers.Dropout(0.2),
    
    layers.Conv1D(128, kernel_size=3, activation='relu'),
    layers.GlobalAveragePooling1D(),
    layers.Dropout(0.2),
    
    layers.Dense(128, activation='relu'),
    layers.Dense(len(np.unique(y)), activation='softmax') # One output per pose
])

model.compile(optimizer='adam', 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

# Training the model
history = model.fit(X_train, y_train, epochs=50, validation_split=0.1, batch_size=32)

# Evaluation
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.4f}")

2026-02-21 17:35:31.571514: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Epoch 1/50


/Users/ankiya/Documents/yoga_pose_corrector/venv/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.0350 - loss: 3.6714 - val_accuracy: 0.0377 - val_loss: 3.6365
Epoch 2/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.0462 - loss: 3.6044 - val_accuracy: 0.0503 - val_loss: 3.5615
Epoch 3/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.0559 - loss: 3.5129 - val_accuracy: 0.0566 - val_loss: 3.4754
Epoch 4/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.0643 - loss: 3.4424 - val_accuracy: 0.0377 - val_loss: 3.4768
Epoch 5/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.0762 - loss: 3.3812 - val_accuracy: 0.0692 - val_loss: 3.3644
Epoch 6/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.0860 - loss: 3.3333 - val_accuracy: 0.0755 - val_loss: 3.3133
Epoch 7/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.1098 - loss: 3.2508 - val_accuracy: 0.1132 - val_loss: 3.2406
Epoch 8/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.1392 - loss: 3.1436 - val_accuracy: 0.1509 - val_loss: 3.

In [2]:
# Updated Model Architecture for better accuracy
model = models.Sequential([
    layers.Input(shape=(X.shape[1], 1)),
    layers.Conv1D(128, 3, activation='relu'),
    layers.BatchNormalization(), # Helps the model learn faster
    layers.MaxPooling1D(2),
    
    layers.Conv1D(256, 3, activation='relu'),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling1D(),
    
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3), # Prevents overfitting
    layers.Dense(128, activation='relu'),
    layers.Dense(len(encoder.classes_), activation='softmax')
])

model.compile(optimizer='adam', 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

# Train for longer (100 epochs)
history = model.fit(X_train, y_train, 
                    epochs=100, 
                    validation_split=0.2, 
                    batch_size=32, 
                    verbose=1)

test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"New Test Accuracy: {test_acc:.4f}")

Epoch 1/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - accuracy: 0.1282 - loss: 3.3268 - val_accuracy: 0.0535 - val_loss: 3.6288
Epoch 2/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.2762 - loss: 2.7212 - val_accuracy: 0.0503 - val_loss: 3.9578
Epoch 3/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.3603 - loss: 2.3230 - val_accuracy: 0.0503 - val_loss: 4.5553
Epoch 4/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.4406 - loss: 1.9666 - val_accuracy: 0.0692 - val_loss: 4.8743
Epoch 5/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.4862 - loss: 1.7647 - val_accuracy: 0.0660 - val_loss: 4.8620
Epoch 6/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.5374 - loss: 1.5463 - val_accuracy: 0.0723 - val_loss: 5.1801
Epoch 7/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.5901 - loss: 1.3618 - val_accuracy: 0.1006 - val_loss: 5.2204
Epoch 8/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.6247 - loss: 1.2826 - val_accuracy: 0.

In [ ]:
# Force save the model and the labels
model.save('yoga_classifier_v1.h5')
import numpy as np
np.save('pose_labels.npy', encoder.classes_)

import os
if os.path.exists('yoga_classifier_v1.h5'):
    print("SUCCESS: yoga_classifier_v1.h5 is now in the folder!")
else:
    print("ERROR: File still not found. Check permissions.")